In [ ]:
import torch, transformers

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
tok = transformers.AutoTokenizer.from_pretrained(model_id)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="cuda"
)

In [ ]:
def get_prompt(text):
    sys = '''
    You are a precise information extraction system for geological and geoscience literature. 
    Your task is to extract events and their temporal relations, plus event argument structure and time bounds, from multi-sentence passages.

    Critical requirements:
    - Be literal and evidence-based. Extract only what the passage explicitly supports.
    - Resolve cross-sentence references and coreference (e.g., "this uplift", "the eruption", "it", "these flows").
    - Use domain-appropriate wording (e.g., "uplift", "intrusion", "orogeny", "basalt flows", "marine transgression", "rift initiation", "glaciation", "deposition", "metamorphism").
    - If an item is not stated or cannot be inferred with high confidence, output null/None for that field.
    - Do NOT invent events, times, or relations.

    Output format must be **valid JSON** matching the schema below, and nothing else (no commentary, no markdown).

    Temporal relations (allowed values ONLY):
    BEFORE, AFTER, DURING, CONTAINS, IDENTITY, EQUALS, OVERLAPS

    Semantics (use these precisely):
    - BEFORE: event1 ends strictly before event2 begins.
    - AFTER:  event1 begins after event2 ends.
    - DURING: event1 occurs entirely within the bounds of event2.
    - CONTAINS: event1 fully contains event2.
    - IDENTITY: event1 and event2 refer to the same event or same time span (treat as synonyms).
    - OVERLAPS: events intersect in time but neither fully contains the other.
    - EQUALS: event1 and event2 have the same time span.

    Time normalization:
    - Prefer explicit ISO if present (e.g., “1998-06” or a year range like "-0120/0010” for BCE/CE when stated).
    - If geological periods/epochs are given (e.g., “Late Cretaceous”), keep them as canonical strings (e.g., "Late Cretaceous") and do not guess numeric Ma unless explicitly provided.
    - If only ordering is known (before/after), set start_time/end_time to null/None.
    - If a bounded interval is stated (e.g., “between 72-66 Ma”), set start_time="72 Ma", end_time="66 Ma" (preserve units/labels exactly as written).

    Event arguments:
    - subject: the agent/undergoer or grammatical subject of the event if stated (e.g., “arc magmatism”, “ice sheets”, “fault block”), else null.
    - object: the patient/theme/goal if stated (e.g., “basalt flows”, "sediments", “crust”), else null.
    - event: a short verb-centric phrase naming the event (e.g., “uplift occurred”, “basalt flows erupted”, “marine sediments were deposited”).
    - start_time/end_time: normalized per rules above (string or null).
    - Keep spans concise and faithful to the text.

    Event identification:
    - Treat each distinct geologic process/change as an event (e.g., initiation, peak, cessation can be separate events if text distinguishes them).
    - Merge exact duplicates (different mentions of the same event/time span) and prefer the most informative wording.

    JSON SCHEMA (must follow exactly):
    {
    "events": [
        {
        "id": "E1",
        "event": "string",
        "subject": "string or null",
        "object": "string or null",
        "start_time": "string or null",
        "end_time": "string or null",
        "evidence_span": "verbatim snippet from the passage indicating this event"
        }
    ],
    "temporal_triples": [
        {
        "event1_id": "E#",
        "temp_relation": "BEFORE|AFTER|DURING|CONTAINS|IDENTITY|EQUALS|OVERLAPS",
        "event2_id": "E#",
        "evidence_span": "verbatim snippet that justifies the relation"
        }
    ]
    }

    Validation rules:
    - Every temporal_triple must reference event ids present in "events".
    - Use stable ids "E1", "E2", ... in order of first appearance.
    - evidence_span must be a short quote (<= 30 words) from the passage.
    - Return at least the events explicitly present; it is acceptable to return an empty temporal_triples array if no justified relation is stated.
    - Output MUST be valid JSON (no trailing commas, no comments).
    '''
    user = '''
    Extract events and temporal relations from the following passage:
    {TEXT}
    '''
    return sys, user.replace("{TEXT}", text)

In [7]:
#extract = "Multiple drill lines along 20km of the Prairie Downs Fault (PDF) were completed in the 2017-2018 exploration season. A total of 6276.6m was drilled for 54 drill holes. The aim of the program was to test 20km of the PDF for base metal mineralisation in tenements E52 and E52. Numerous drill holes intersected significant base metal, vanadium and gold mineralisation including 19m @ 5.9% Pb, 0.1% Zn, 0.1% Cu and 40 g/t Ag from 87m in hole PDP456, at the Husky South prospect. Down hole total electro magnetics was completed on the two diamond drill holes PDD504 and PDD506 at Husky South. No significant off hole responses were detected. "
extract = "In 2019, BHP found a rock formation characterised by traces of pyrite. BHP transported the rock to Perth in 2020. After the transport, BHP discovered a quartz and gold vein."
system, user = get_prompt(extract)


messages = [
  {"role":"system","content":system},
  {"role":"user","content":user}
]

inputs = tok.apply_chat_template(messages, add_generation_prompt=True,
                                 return_tensors="pt").to(model.device)
out = model.generate(inputs, max_new_tokens=1000, temperature=0.2, do_sample=False)

# Only decode the generated continuation (exclude the prompt tokens)
gen_tokens = out[0, inputs.shape[-1]:]
print(tok.decode(gen_tokens, skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Here is the extracted information in the required JSON format:

```
{
    "events": [
        {
            "id": "E1",
            "event": "BHP found a rock formation characterised by traces of pyrite",
            "subject": "BHP",
            "object": "rock formation",
            "start_time": "2019",
            "end_time": null,
            "evidence_span": "In 2019, BHP found a rock formation characterised by traces of pyrite."
        },
        {
            "id": "E2",
            "event": "BHP transported the rock to Perth",
            "subject": "BHP",
            "object": "rock",
            "start_time": "2020",
            "end_time": null,
            "evidence_span": "BHP transported the rock to Perth in 2020."
        },
        {
            "id": "E3",
            "event": "BHP discovered a quartz and gold vein",
            "subject": "BHP",
            "object": "quartz and gold vein",
            "start_time": null,
            "end_time": null,
            "

In [ ]:
decodings = tok.decode(out[0], skip_special_tokens=True)

In [ ]:
decodings